# Diffusion model training:

In [1]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST, QMNIST, KMNIST
from torch.utils.data import DataLoader, TensorDataset
from src.dataset import LatentBatchDataset, ShardedLatentDataset
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time
import json
from tqdm import tqdm  # import the class directly
from datetime import timedelta

import torchaudio.transforms as T
from src.dataset import NSynth
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
import math
from src.paths import PATHS, VAE_CHANNELS, VAE_LATENT_DIM, VAE_STRIDES
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def show_loss_plot(losses):
    epochs_list   = [d['epoch']    for d in losses]
    loss_list     = [d['loss']     for d in losses]
    val_loss_list = [d['val_loss'] for d in losses]
    lr_list       = [d['lr']       for d in losses]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    ax1.plot(epochs_list, loss_list,     label='Train loss')
    if any(v is not None for v in val_loss_list):
        ax1.plot(epochs_list, val_loss_list, label='Val loss', color='red', linestyle='--')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs_list, lr_list, color='orange')
    ax2.set_ylabel('Learning Rate')
    ax2.set_xlabel('Epoch')
    ax2.set_title('Learning Rate Schedule')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

Every diffusion model is trained in a similar way:

For each image extracted from the dataset, we add a random amount of noise to it and we ask the model to predict the noise that we added. The loss is the mean squared error between the predicted noise and the actual noise.

In [2]:
def save_checkpoint(model, scheduler, epoch, losses, path):
    torch.save({
        'epoch': epoch,
        'model': model,
        'scheduler': scheduler,
        'losses': losses,
    }, path)

def load_checkpoint(model, scheduler, path):
    checkpoint = torch.load(path, map_location=device)
    
    model = checkpoint['model']
    scheduler = checkpoint['scheduler']
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
    
    return model, scheduler, checkpoint['epoch'], checkpoint['losses']

def train(
    model, 
    scheduler,
    transform_data=None,
    epochs=100, 
    batch_size=16, 
    lr=1e-3, 
    modality='diffusion',
    dataset=NSynth('training'), 
    dataset_val=None,
    patience=6,
    resume=False,
    resume_lr = True,
    autocast=True,
    shuffle=True,
    num_workers=1
):
    
    print('Training started...')
    model.train()

    validation = dataset_val is not None
    epochs_without_improvement = 0
    if validation:
        valid_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=num_workers)
    
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=num_workers)
    diffuser = LatentDiffuser(model, scheduler).to(device) if modality == 'vae_latent' or modality == 'encoder_latent' else Diffuser(model, scheduler).to(device)
    # lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    mse_loss = nn.MSELoss()
    best_loss = 1
    start_epoch = 0
    losses = []
    
    num_timesteps = scheduler.alpha_bar.shape[0]


    # Retomar desde checkpoint si existe
    if resume and os.path.exists(PATHS[modality]['checkpoint']):
        model, scheduler, start_epoch, losses = load_checkpoint(
            model, scheduler, PATHS[modality]['checkpoint']
        )
        best_loss = min(losses, key=lambda x: x['loss'])['loss']
        start_epoch += 1  # continúa desde el siguiente epoch
        if resume_lr:
            lr = losses[-1]['lr']
        
        print(f"Retomando desde epoch {start_epoch}, mejor loss: {best_loss:.4f}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    
    scaler = torch.cuda.amp.GradScaler()
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs - start_epoch
    )
    
    start_time = time.time() 
    for epoch in range(start_epoch, epochs):
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        epoch_loss = 0
        for x in pbar:
            # wave = wave.to(device)
            # x = stft_transform(wave)
            if transform_data is not None:
                x = transform_data(x)
            x = x.to(device)                
            B = x.size(0)
            t = torch.randint(0, num_timesteps, (B,), device=device, dtype=torch.long)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=autocast): # OPTIMIZACION FLOAT 32
                # print(f"mag: {mag.shape}, sin: {sin.shape}, cos: {cos.shape}, x: {x.shape}")
                z, e = diffuser(x, t)
                e_pred = model(z, t)
                if e_pred.shape != e.shape: # TODO no entiendo muy bien que ocurre
                    print('Error en las shapes')
                    e_pred = e_pred.squeeze(1)  # [B, 1, H, W] → [B, H, W]
                loss = mse_loss(e_pred, e)
                epoch_loss += loss.item()

            scaler.scale(loss).backward() # TODO no se si la funcion de loss es la mas optima para el caso
            scaler.step(optimizer)

            scaler.update()
            
            pbar.set_postfix({'batch loss': loss.item(), 'avg loss': epoch_loss /( pbar.n + 1)})

            # FIN ENTRENAMIENTO
            
        # VALIDACION
        running_val_loss = 0.
        if validation:
            vpbar = tqdm(valid_loader, desc=f"Validation {epoch+1}/{epochs}", unit="batch")
            model.eval()
            with torch.no_grad():
                for x in vpbar:
                    if transform_data is not None:
                        x = transform_data(x)
                    x = x.to(device)
                    B = x.size(0)
                    t = torch.randint(0, num_timesteps, (B,), device=device, dtype=torch.long)
                    z, e = diffuser(x, t)
                    e_pred = model(z, t)
                    if e_pred.shape != e.shape:
                        e_pred = e_pred.squeeze(1)
                    loss = mse_loss(e_pred, e)
                    running_val_loss += loss.item()
                    vpbar.set_postfix({'batch val loss': loss.item(), 'avg val loss': running_val_loss / (vpbar.n + 1)})
            model.train()
            
        avg_loss = epoch_loss / len(train_loader)
        avg_val_loss = running_val_loss / len(valid_loader) if validation else None
        
        losses.append({
            'epoch': epoch,
            'loss': avg_loss,
            'lr': optimizer.param_groups[0]['lr'],
            'time' : time.time() - start_time,
            'val_loss': avg_val_loss
        })        
        
        lr_scheduler.step() # LEARNING SCHEDULER
            
                
        reference = avg_val_loss if validation else avg_loss
        if reference < best_loss:
            best_loss = avg_loss
            epochs_without_improvement = 0
            if modality:
                # print('Checkpoint guardado')
                save_checkpoint(model, scheduler, epoch, losses, PATHS[modality]['checkpoint'])
                json.dump(model.get_config(), open(PATHS[modality]['config'], 'w'))
          
        # Early stopping cada x epochs sin mejora
        if validation:
            print(f'Training loss: {avg_loss},\t Validation loss: {avg_val_loss}')
            if avg_val_loss >= best_loss:
                epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

    # AL FINALIZAR EL ENTRENAMIENTO, GUARDAMOS EL MODELO Y EL SCHEDULER
    if modality:
        model, scheduler, _, _= load_checkpoint(model, scheduler, PATHS[modality]['checkpoint'])
        torch.save(model, PATHS[modality]['model'])
        torch.save(scheduler, PATHS[modality]['scheduler'])
        json.dump(model.get_config(), open(PATHS[modality]['config'], 'w'))
        
    _t = time.time() - start_time
    print(f"Training completed, best loss: {best_loss}, total time: {str(timedelta(seconds=int(_t)))}")
    
    return model, scheduler, losses
    

## MINST

In [3]:
%%writefile ../src/diffusion_setups/setup_minst.py

import torch
import torch.nn as nn

from src.diffusion import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def MINST_transform(x):
    img, _ = x
    return img

def setup_minst_model(timesteps=1000, channels=32, norm_groups=6, emb_dim=128):
    ## Image Size
    # input_height = 28
    # input_width = 28
    # input_size = (input_height, input_width)

    config = {
        'timesteps': timesteps,
        'channels': channels,
        'norm_groups': norm_groups,
        'emb_dim': emb_dim
    }

    down_layers = [  
        DummyLayer(channels,    channels*2,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c canales
        DummyLayer(channels*2,  channels*4,  norm_groups, emb_dim, skip=False, stride=1).to(device),  # skip: c*2 canales
        DummyLayer(channels*4,  channels*8,  norm_groups, emb_dim, skip=False, stride=1).to(device),  # skip: c*4 canales
        # DummyLayer(c*8,  c*16, norm_groups, emb_dim, skip=True, stride=2).to(device),  # skip: c*8 canales
    ]

    bottleneck = DummyLayer(channels*8, channels*8, norm_groups, emb_dim).to(device)

    up_layers = [
        # DummyLayer(c*16 + c*8,  c*8, norm_groups, emb_dim, stride=-2).to(device),
        DummyLayer(channels*8,  channels*4, norm_groups, emb_dim, stride=-1).to(device),
        DummyLayer(channels*4,  channels*2, norm_groups, emb_dim, stride=-1).to(device),
        DummyLayer(channels*2 + channels,    channels,   norm_groups, emb_dim, stride=-1).to(device),
    ]

    # EMBEDDER
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # SCHEDULER
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    # MODEL
    model = DiffusionModel(
        layer_channels=(channels, channels),
        norm_groups=norm_groups,
        up_layers=up_layers,
        down_layers=down_layers,
        bottleneck=bottleneck,
        embedder=embedder, 
        input_channels=1,
        output_channels=1,
        config=config
    ).to(device)
    
    return model, scheduler, MINST_transform

Overwriting ../src/diffusion_setups/setup_minst.py


In [4]:
def train_MINST(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=2048, 
    learning_rate=1e-4, 
    resume=False
    ):

    # %%time
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality='minst_VAE',
        dataset= QMNIST(root=PATHS['minst_VAE']['dataset'], train=True,  download=True, transform=ToTensor()),
        resume=resume,
        resume_lr=True
    )
    show_loss_plot(losses)
    
    return model, scheduler, losses


## AUDIO

In [5]:
# %%writefile ../src/diffusion_setups/setup_audio.py

import torch
import torch.nn as nn

from src.diffusion import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
import torchaudio.transforms as T


class AudioPipeline:
    def __init__(self, stft_transform):
        self.stft_transform = stft_transform

    def __call__(self, x):
        wave, _, _, _ = x
        wave = wave.to(device)  # <-- mover wave a GPU antes de la STFT
        stft_spec = self.stft_transform(wave)
        del wave
        log_mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec)
        return torch.cat([log_mag, sin, cos], dim=1)  # ya está en device

def setup_audio_model(timesteps=1000, channels=32, norm_groups=6, emb_dim=12, n_fft=1500, hop_length=250, win_length=1500):
    ''' Devuelve el modelo de audio listo para entrenar'''
    
    config = {
        'timesteps': timesteps,
        'channels': channels,
        'norm_groups': norm_groups,
        'emb_dim': emb_dim,
        'n_fft': n_fft,
        'hop_length': hop_length,
        'win_length': win_length
    }
    
    
    sample_rate = 16000
    # n_fft = 1500 # DISMINUIR TAMAÑO PARA OPTIMIZAR
    # hop_length = 250
    # win_length = n_fft
    
    sample_rate = 16000
    # n_fft = 1024//2
    # hop_length = 256
    # win_length = 1024//2
    
    # Data pipeline
    stft_transform = T.Spectrogram(
        n_fft=n_fft,
        win_length=win_length, 
        hop_length=hop_length,
        power=None, 
        onesided=True,
        center=False
    ).to(device)
    
    
    transform_audio = AudioPipeline(stft_transform)
    
    # Layers
    down_layers = [  
        DummyLayer(channels,    channels*2,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c canales
        DummyLayer(channels*2,  channels*4,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c*2 canales
    ]

    bottleneck = DummyLayer(channels*4, channels*4, norm_groups, emb_dim).to(device)

    up_layers = [ # TODO Podria hacer esto con stride = -2? o coger y hacer que si el stride es negativo, aumente manualmente 
        DummyLayer(channels*4 + channels*2, channels*2, norm_groups, emb_dim, stride=1).to(device),
        DummyLayer(channels*2 + channels, channels, norm_groups, emb_dim, stride=1).to(device),
    ]

    # Embedder
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # scheduler
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    # model
    model = DiffusionModel(
        layer_channels=(channels, channels), # Por ahora los canales de entrada y salida son los mismos
        norm_groups=norm_groups,
        up_layers=up_layers,
        down_layers=down_layers,
        bottleneck=bottleneck,
        embedder=embedder, 
        input_channels=3, ##SINCOS
        output_channels=3, ##SINCOS
        config=config
    ).to(device)
    
    transform_audio = AudioPipeline(stft_transform)
    
    return model, scheduler, transform_audio

In [6]:
# FULL DIFFUSION - NO EJECUTAR
def train_audio(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=128, 
    learning_rate=1e-4, 
    resume=False
    ):
    
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality='diffusion',
        dataset=NSynth('training'),
        resume=resume
    )
    show_loss_plot(losses)
    return model, scheduler, losses

# LATENT SPACE
We are going to train a diffusion model in the latent space of a VAE and an AutoEncoder.

In [7]:
%%writefile ../src/diffusion_setups/setup_latent.py

import torch
import torch.nn as nn

from src.diffusion import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def latent_pipeline(x):
    if isinstance(x, (list, tuple)):
        x = x[0]        # TensorDataset envuelve en tupla → [B, C, H, W]
    return x             # latente espacial del VAE, ya en la forma correcta

def setup_latent_model(timesteps=1000, emb_dim=128, hidden_dims=32, latent_dim=8,
                        norm_groups=6, deep=4, increase_dims=True):
    '''Devuelve el modelo de difusión latente (2D, sobre el espacio del VAE) listo para entrenar'''

    config = {
        'timesteps': timesteps,
        'emb_dim': emb_dim,
        'hidden_dims': hidden_dims,
        'latent_dim': latent_dim,
        'norm_groups': norm_groups,
        'deep': deep,
        'increase_dims': increase_dims,
    }

    if increase_dims:
        ch_progression = [hidden_dims * (2 ** i) for i in range(deep)]  # [32, 64, 128, 256]
    else:
        ch_progression = [hidden_dims for _ in range(deep)]             # [32, 32, 32, 32]

    ######################
    #### OPCION 1 ########
    ######################

    # Down: hidden_dims -> ch_progression[0] -> ch_progression[1] -> ...
    # down_layers = []
    # in_ch = hidden_dims
    # for out_ch in ch_progression:
    #     down_layers.append(DummyLayer(in_ch, out_ch, norm_groups, emb_dim, skip=True, stride=-2).to(device))
    #     in_ch = out_ch
    # down_layers = nn.ModuleList(down_layers)

    # bottleneck = DummyLayer(in_ch, in_ch, norm_groups, emb_dim).to(device)

    # # Up: espejo del down, sumando los canales del skip correspondiente
    # skip_channels = [hidden_dims] + ch_progression[:-1]
    # up_layers = []
    # for out_ch in reversed(skip_channels):
    #     up_layers.append(DummyLayer(in_ch + out_ch, out_ch, norm_groups, emb_dim, stride=-2).to(device))
    #     in_ch = out_ch
    # up_layers = nn.ModuleList(up_layers)
    
    ######################
    #### OPCION 2 ########
    ######################
    
    # Down
    down_layers = []
    in_ch = hidden_dims
    for i, out_ch in enumerate(ch_progression):
        stride = 1 if i == 0 else 1      # ↓ reduce resolución
        down_layers.append(
            DummyLayer(
                in_ch,
                out_ch,
                norm_groups,
                emb_dim,
                skip=True,
                stride=stride
            ).to(device)
        )
        in_ch = out_ch

    down_layers = nn.ModuleList(down_layers)

    bottleneck = DummyLayer(
        in_ch,
        in_ch,
        norm_groups,
        emb_dim,
        stride=1
    ).to(device)

    # Up
    skip_channels = [hidden_dims] + ch_progression[:-1]
    up_layers = []

    for i, out_ch in enumerate(reversed(skip_channels)):
        stride = 1 if i < len(skip_channels)-1 else 1   # ↑ aumenta resolución
        up_layers.append(
            DummyLayer(
                in_ch + out_ch,
                out_ch,
                norm_groups,
                emb_dim,
                stride=stride
            ).to(device)
        )
        in_ch = out_ch

    up_layers = nn.ModuleList(up_layers)

    # Embedder
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # Scheduler
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    model = DiffusionModel(
        layer_channels=(hidden_dims, hidden_dims),  # conv_in/conv_out: latent_dim <-> hidden_dims
        norm_groups=norm_groups,
        down_layers=down_layers,
        up_layers=up_layers,
        bottleneck=bottleneck,
        embedder=embedder,
        input_channels=latent_dim,
        output_channels=latent_dim,
        config=config
    ).to(device)

    return model, scheduler, latent_pipeline

Overwriting ../src/diffusion_setups/setup_latent.py


In [8]:
def train_latent(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=128, 
    learning_rate=1e-4, 
    resume=False,
    source='vae_diffusion'
    ):
    
    # latents = torch.load(rf"{PATHS[source]['dataset_training']}")
    # val_latents = torch.load(rf"{PATHS[source]['dataset_validation']}")
        
    # Calcula media y std del dataset de latentes
    # mean = latents.mean()
    # std = latents.std()
    # latents = (latents - mean) / std  # que tengan distribución ~N(0,1)
    
    dataset = ShardedLatentDataset(PATHS[source]['dataset_training'], shuffle=True, shard_buffer=3)
    dataset_val=ShardedLatentDataset(PATHS[source]['dataset_validation'], shuffle=True, shard_buffer=3)

    
    # dataset=LatentBatchDataset(PATHS[source]['dataset_training'])
    # dataset_val=LatentBatchDataset(PATHS[source]['dataset_validation'])

    
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality=source,
        dataset=dataset,
        dataset_val=dataset_val,
        patience=10,
        resume=resume,
        autocast=False,
        num_workers=4,
        shuffle=False
    )
    show_loss_plot(losses)
    return model, scheduler, losses

# MODEL TRAINING

In [9]:
# ''' 
# ENTRENAMIENTO DEL MODELO MINST
#     se pueden configurar más elementos directamente en setup_minst_model 

# '''
# from src.diffusion_setups.setup_minst import setup_minst_model

# CHECKPOINT = True

# model, scheduler, trans = setup_minst_model(
#     timesteps=500, # Tiene que ser el mismo que la que tuviera el modelo
#     channels=64,
#     norm_groups=8,
#     emb_dim=128
# )

# model, scheduler, losses = train_MINST(
#     model=model,
#     scheduler=scheduler,
#     transform_data=trans,
#     epochs=10,
#     batch_size=2048//8,
#     learning_rate=1e-3,
#     resume=CHECKPOINT
# )

In [10]:
# ''' 
# ENTRENAMIENTO DEL MODELO AUDIO
#     se pueden configurar más elementos directamente en setup_audio_model 

# '''
# from src.diffusion_setups.setup_audio import setup_audio_model


# CHECKPOINT = False

# model, scheduler, trans = setup_audio_model(
#     timesteps=750, # Tiene que ser el mismo que la que tuviera el modelo
#     channels=32*4,
#     norm_groups=8*4,
#     emb_dim=128*4,
#     n_fft = 1024//2,
#     hop_length = 1024//4,
#     win_length = 1024//2,
# )


# # model, scheduler, losses = train_audio(
# #     model=model,
# #     scheduler=scheduler,
# #     transform_data=trans,
# #     epochs=3,
# #     batch_size=8,
# #     learning_rate=1e-3,
# #     resume=CHECKPOINT
# # )

In [11]:
''' 
ENTRENAMIENTO DEL MODELO LATENT DIFFUSION CON EL VAE
    se pueden configurar más elementos directamente en setup_latent_model 

'''
CHECKPOINT = False

from src.diffusion_setups.setup_latent import setup_latent_model


model, scheduler, trans = setup_latent_model(
    timesteps=1000, 
    emb_dim=128, 
    norm_groups=8,
    hidden_dims=32, 
    latent_dim=32, # viene dado por el VAE
    deep=4, 
    increase_dims=True
)

model, scheduler, losses = train_latent(
    model=model,
    scheduler=scheduler,
    transform_data=trans,
    epochs=15,
    batch_size=2048//8,
    learning_rate=1e-3,
    resume=CHECKPOINT,
    source='vae_diffusion'
)
# Training completed, best loss: 0.272, total time: 0:23:00

c:\Users\Articuno\Desktop\TFG-MUSICAL\env-musical-serious-business\Lib\site-packages\torch\_utils.py:831: UserWarning:

TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()



Training started...


Epoch 1/15:   0%|          | 0/1130 [00:12<?, ?batch/s]


RuntimeError: Given groups=1, weight of size [32, 32, 3, 3], expected input[256, 64, 94, 63] to have 32 channels, but got 64 channels instead

In [ ]:
''' 
ENTRENAMIENTO DEL MODELO LATENT DIFFUSION CON EL AUTOENCODER
    se pueden configurar más elementos directamente en setup_latent_model 

'''
CHECKPOINT = False

from src.diffusion_setups.setup_latent import setup_latent_model


model, scheduler, trans = setup_latent_model(
    timesteps=1000, 
    emb_dim=256, 
    hidden_dims=512, 
    latent_dim=200, # siempre es 200 porque es el tamaño del vector del VAE
    deep=3, 
    increase_dims=False
)

model, scheduler, losses = train_latent(
    model=model,
    scheduler=scheduler,
    transform_data=trans,
    epochs=200,
    batch_size=2048//16,
    learning_rate=1e-3,
    resume=CHECKPOINT,
    source='autoencoder_diffusion'
)

# [EXTRA] Extract all VAE and encoder latents
Using the trained VAE, we can extract the latent representations of the audio samples in the dataset. This will allow us to train a diffusion model on these latent representations.

In [4]:
%run init_notebook.py

import torch.nn as nn
from torch.utils.data import DataLoader
import json
from tqdm import tqdm  # import the class directly

import torchaudio.transforms as T
from src.dataset import NSynth
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
import math
from src.paths import PATHS, VAE_CHANNELS, VAE_LATENT_DIM, VAE_STRIDES
from src.latent_models import latent_VAE, latent_AutoEncoder
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_vae(input_height=1500, input_width=251, latent_dim=8, channels=[2,16,32,64], model_path=PATHS['VAE_2D']['model']):
    model = latent_VAE(input_size=(input_height, input_width), latent_dim=VAE_LATENT_DIM, strides=VAE_STRIDES, channels=VAE_CHANNELS).to(device)

    model.load_state_dict(torch.load(model_path))
    model.eval()
    print("VAE Loaded")
    return model

def load_encoder(input_height=4000, input_width= 201, latent_dim=8, channels = [2, 16, 32, 64], model_path=PATHS['autoencoder_2D']['model']):
    model = latent_AutoEncoder((input_height, input_width), latent_dim, channels).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    print("Encoder Loaded")
    return model

# def extract_latents(model, dataset, stft_transform, variational=True):
#     dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
#     all_z = []
#     pbar = tqdm(dataloader, desc="Extracting latents")
#     with torch.no_grad():
#         for waveform, _, _, _ in pbar:
#             waveform = waveform.to(device)
#             stft_spec = stft_transform(waveform)
#             # log_mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec)
#             # x = torch.cat([log_mag, sin, cos], dim=1).to(device)
#             log_mag, phase = compute_magnitude_and_phase(stft_spec)
#             x = torch.cat([log_mag, phase], dim=1).to(device)
#             if variational:
#                 feat, mu, logvar = model.encoder(x)
#                 z = mu # TODO no se si ignorar el resto de cosas es lo mejor...
#                 # z = model.reparameterize(mu, logvar) # TODO probar con y sin esto
#             else:
#                 z = model.encoder(x)
#             all_z.append(z.cpu().half()) # a la mitad (float16) porque no cabe en la ram #TODO, guardar por batch
#     all_z = torch.cat(all_z, dim=0)    
#     return all_z


# def extract_latents(model, dataset, stft_transform, save_dir, variational=True, batch_size=256):
#     os.makedirs(save_dir, exist_ok=True)
#     dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

#     pbar = tqdm(dataloader, desc="Extracting latents")
#     batch_idx = 0
#     n_samples = 0
#     sum_ = 0
#     sum_sq = 0
#     count = 0

#     with torch.no_grad():
#         for waveform, _, _, _ in pbar:
#             waveform = waveform.to(device)
#             stft_spec = stft_transform(waveform)
#             log_mag, phase = compute_magnitude_and_phase(stft_spec)
#             x = torch.cat([log_mag, phase], dim=1).to(device)

#             if variational:
#                 feat, mu, logvar = model.encoder(x)
#                 z = mu
#             else:
#                 z = model.encoder(x)

#             z = z.cpu()
#             torch.save(z, os.path.join(save_dir, f"batch_{batch_idx:05d}.pt"))
            
#             sum_ += z.sum().item()
#             sum_sq += (z.double() ** 2).sum().item()
#             count += z.numel()
            
#             batch_idx += 1
#             n_samples += z.shape[0]

#     print(f"Guardados {batch_idx} batches ({n_samples} muestras) en {save_dir}")
#     mean = sum_ / count
#     std = (sum_sq / count - mean ** 2) ** 0.5
    
#     stats_path = os.path.join(save_dir, "stats.json")
#     with open(stats_path, "w") as f:
#         json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)
    
#     return batch_idx, n_samples, mean, std

import threading
import queue

def extract_latents(model, dataset, stft_transform, save_dir, variational=True, batch_size=256, save_queue_size=8):
    os.makedirs(save_dir, exist_ok=True)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    save_q = queue.Queue(maxsize=save_queue_size)

    def saver_worker():
        while True:
            item = save_q.get()
            if item is None:
                save_q.task_done()
                break
            tensor, path = item
            torch.save(tensor, path)
            save_q.task_done()

    saver_thread = threading.Thread(target=saver_worker, daemon=True)
    saver_thread.start()

    pbar = tqdm(dataloader, desc="Extracting latents")
    batch_idx = 0
    n_samples = 0
    sum_ = 0
    sum_sq = 0
    count = 0

    with torch.no_grad():
        for waveform, _, _, _ in pbar:
            waveform = waveform.to(device)
            stft_spec = stft_transform(waveform)
            log_mag, phase = compute_magnitude_and_phase(stft_spec)
            x = torch.cat([log_mag, phase], dim=1).to(device)

            if variational:
                feat, mu, logvar = model.encoder(x)
                z = mu
            else:
                z = model.encoder(x)

            z = z.to(torch.float16).cpu()
            save_q.put((z, os.path.join(save_dir, f"batch_{batch_idx:05d}.pt")))

            sum_ += z.sum().item()
            sum_sq += (z.double() ** 2).sum().item()
            count += z.numel()

            batch_idx += 1
            n_samples += z.shape[0]

    # esperar a que el hilo termine de volcar todo a disco
    save_q.put(None)
    saver_thread.join()
    
    mean = sum_ / count
    std = (sum_sq / count - mean ** 2) ** 0.5
    
    stats_path = os.path.join(save_dir, "stats.json")
    
    with open(stats_path, "w") as f:
        json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)

    return n_samples, sum_, sum_sq, count

def extract_stats(save_dir):
    ''' No es necesario ejecutar esto si ya se ha ejecutado extract_latents, porque ya guarda los stats en stats.json '''
    batch_files = sorted(
        os.path.join(save_dir, f) for f in os.listdir(save_dir)
        if f.startswith("batch_") and f.endswith(".pt")
    )

    sum_ = 0.0
    sum_sq = 0.0
    count = 0
    n_samples = 0

    for fname in tqdm(batch_files, desc="Computing stats"):
        z = torch.load(fname, map_location='cpu').double()  # subir a double para evitar overflow/error de precision
        sum_ += z.sum().item()
        sum_sq += (z ** 2).sum().item()
        count += z.numel()
        n_samples += z.shape[0]

    mean = sum_ / count
    std = (sum_sq / count - mean ** 2) ** 0.5

    stats_path = os.path.join(save_dir, "stats.json")
    with open(stats_path, "w") as f:
        json.dump({"mean": mean, "std": std, "n_samples": n_samples}, f)

    print(f"Stats guardadas en {stats_path}: mean={mean:.4f}, std={std:.4f}, n_samples={n_samples}")
    return mean, std

def normalize_latents(save_dir):
    stats_path = os.path.join(save_dir, "stats.json")
    with open(stats_path, "r") as f:
        stats = json.load(f)
    mean = stats["mean"]
    std = stats["std"]

    batch_files = sorted(
        os.path.join(save_dir, f) for f in os.listdir(save_dir)
        if f.startswith("batch_") and f.endswith(".pt")
    )

    for fname in tqdm(batch_files, desc="Normalizing batches"):
        z = torch.load(fname, map_location='cpu')
        z = (z - mean) / std
        torch.save(z, fname)

    stats["normalized"] = True
    with open(stats_path, "w") as f:
        json.dump(stats, f)

    print(f"Normalizados {len(batch_files)} batches en {save_dir}")

Latents are extracted by passing the spectrograms through the encoder part of the VAE.

In [ ]:
modes = ['validation', 'training']
variational = True
model_str = 'vae_diffusion' if variational else 'autoencoder_diffusion'

model = load_vae(latent_dim=32, channels=[2, 16, 32, 64, 128]) if variational else load_encoder()

n_fft = 1500 # DISMINUIR TAMAÑO PARA OPTIMIZAR
hop_length = 250
win_length = n_fft

stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=None, onesided=True, center=False
).to(device)

for mode in modes:
    print(f"Processing {mode} set...")
    dataset = NSynth(mode)
    extract_latents(model, dataset, stft_transform, save_dir=PATHS[model_str][f'dataset_{mode}'], variational=variational)
    # extract_stats(save_dir=PATHS[model_str][f'dataset_{mode}'], )
    normalize_latents(save_dir=PATHS[model_str][f'dataset_{mode}'])

VAE Loaded
Processing test set...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Articuno\\Desktop\\TFG-MUSICAL\\data\\test\\examples.json'